# V7_A_N10 — Satellites Need Ground Truth

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Synthetic training data. Decision support only; specialist and institutional review required.

## Decision contract
Combine earth observation with field evidence for crop-area and condition review. Satellite classifications do not establish ownership, compliance, or official production totals by themselves.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7610);n=600
df=pd.DataFrame({'zone':rng.choice(['North','Central','South'],n),'ndvi':rng.uniform(.1,.85,n),'rain':rng.uniform(20,180,n),'cloud':rng.uniform(0,1,n),'field_crop':rng.choice([0,1],n,p=[.58,.42])});df['pixel_ha']=10;df.head()

## Evidence and quality contract
Record sensor/product, acquisition date, resolution, processing version, cloud mask, classification legend, boundary version, and validation sample design.

In [2]:
df['usable']=df.cloud<.35;df['pred_crop']=((df.ndvi>.43)&(df.rain>55)).astype(int);v=df[df.usable];cm=pd.crosstab(v.field_crop,v.pred_crop);acc=(v.field_crop==v.pred_crop).mean();print(cm.to_string());print('usable_share',round(df.usable.mean(),3),'accuracy',round(acc,3))

pred_crop    0   1
field_crop        
0           81  42
1           54  39
usable_share 0.36 accuracy 0.556


## Uncertainty, sensitivity, and abstention
Clouds and classification errors propagate into area estimates. Report raw mapped area with a simple error-adjusted sensitivity range; abstain where usable coverage is low.

In [3]:
z=v.groupby('zone').agg(mapped_ha=('pred_crop',lambda x:10*x.sum()),pixels=('pred_crop','size'),accuracy=('pred_crop',lambda x:(x==v.loc[x.index,'field_crop']).mean())).reset_index();z['lower_ha']=z.mapped_ha*(z.accuracy-.1).clip(lower=0);z['upper_ha']=z.mapped_ha*(z.accuracy+.1).clip(upper=1);z['status']=np.where(z.accuracy<.55,'ABSTAIN—FIELD VALIDATION','AREA ESTIMATE REVIEW');result=z;print(result.round(1).to_string(index=False))

   zone  mapped_ha  pixels  accuracy  lower_ha  upper_ha                   status
Central        390      85       0.6     185.8     263.8     AREA ESTIMATE REVIEW
  North        220      63       0.5      82.8     126.8 ABSTAIN—FIELD VALIDATION
  South        200      68       0.6     100.6     140.6     AREA ESTIMATE REVIEW


## Decision product and authority boundary
The output is a review queue with reason codes, evidence age, uncertainty, accountable owner, and status. It never transfers legal, clinical, professional, procurement, enforcement, or budget authority to the model.

## Exercises
1. Stratify validation sampling. 2. Explain resolution limits. 3. Add a change-detection check.

## Exact solutions
1. Sample across zones, crop prevalence, terrain, and confidence classes with known inclusion probabilities. 2. Mixed pixels and small fields limit interpretation below pixel scale. 3. Hold legend and processing constant, quantify classification uncertainty, and verify apparent change in the field.

In [4]:
assert len(result)>0 and result['status'].notna().all(); print('V7_A_N10_COMPLETE_EXECUTION_PASS')

V7_A_N10_COMPLETE_EXECUTION_PASS
